In [5]:
import re
import numpy as np
import pandas as pd
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
from pathlib import Path
from IPython.display import display

In [48]:
input_path = Path("ARR_MARCH") / "Output" / "summaries" / "CC_VE" 

In [7]:
single_culture = {"Armenia", "Germany", "Greece", "Japan", "Netherlands"}
multi_culture  = {"Colombia", "Malaysia", "Mexico", "Peru", "United_States"}

In [8]:
csv_files = [
    f for f in sorted(input_path.rglob("*.csv"))
    if "checkpoint" not in f.name.lower()
    and ".ipynb_checkpoints" not in str(f).lower()
]

In [9]:
def infer_domain_from_path(p: Path) -> str:
    stem = p.stem  
    stem = re.sub(r"[-_]?checkpoint$", "", stem, flags=re.IGNORECASE)  
    
    parts = stem.split("_")
    roman = {"I", "II", "III"}

    if len(parts) >= 2 and parts[-1].upper() in roman:
        return f"{parts[-2].upper()}_{parts[-1].upper()}"
    return parts[-1].upper()

In [10]:
required_cols = {"domain", "country", "group", "variance_explained", "pct_agreement_consensus"}

In [11]:
df_list = []
for f in csv_files:
    df = pd.read_csv(f, encoding="utf-8-sig")
    df.columns = [c.strip() for c in df.columns]  
    if not required_cols.issubset(df.columns):
        print(f"Skipping {f.name} (missing cols: {sorted(required_cols - set(df.columns))})")
        continue

    df["domain"] = infer_domain_from_path(f)
    df["file"] = f.name  
    df_list.append(df)

In [12]:
df_all = pd.concat(df_list, ignore_index=True)

In [13]:
def collapse_domain(d):
    # SCTOM_I/II/III -> SCTOM
    d = re.sub(r"^SCTOM_(I|II|III)$", "SCTOM", d)
    # (Optional) also collapse these if you want:
    d = re.sub(r"^SVNS_(I|II|III)$", "SVNS", d)
    d = re.sub(r"^PCPR_(I|II)$", "PCPR", d)
    d = re.sub(r"^PIPP_(I|II)$", "PIPP", d)
    return d

In [14]:
df_all["domain_grp"] = df_all["domain"].apply(collapse_domain)

In [15]:
selected_cols = ["domain_grp", "country", "group", "variance_explained", "pct_agreement_consensus"]

In [16]:
df_all = df_all[selected_cols]

In [17]:
df_CC = (df_all
              .groupby(["country", "domain_grp"], as_index=False)["pct_agreement_consensus"]
              .mean())

In [34]:
rows_agreement = []
for dom, sub in df_CC.groupby("domain_grp"):

    if sub["pct_agreement_consensus"].max() > 1.0:
        sub["pct_agreement_consensus"] = sub["pct_agreement_consensus"] / 100.0
    else:
        sub["pct_agreement_consensus"] = sub["pct_agreement_consensus"]
        
    def culture_of(c):
        if c in single_culture:
            return "Single"
        if c in multi_culture:
            return "Multi"
        return np.nan

    sub["culture"] = sub["country"].map(culture_of)
    
    x = sub.loc[sub["culture"].eq("Multi"), "pct_agreement_consensus"].dropna().to_numpy()
    y = sub.loc[sub["culture"].eq("Single"), "pct_agreement_consensus"].dropna().to_numpy()

    t, p = ttest_ind(x, y, equal_var=False, nan_policy="omit")
    rows_agreement.append({
            "Domain": dom,
            "Mean (Multi)": np.mean(x) if len(x) else np.nan,
            "Mean (Single)": np.mean(y) if len(y) else np.nan,
            "Δ (M−S)": (np.mean(x) - np.mean(y)) if (len(x) and len(y)) else np.nan,
            "t": t,
            "p": p
    })

dfagreement = pd.DataFrame(rows_agreement).sort_values("Domain").reset_index(drop=True)
dfagreement["q_FDR"] = multipletests(dfagreement["p"], method="fdr_bh")[1]

C:\Users\kpoth\anaconda3\envs\CCT_env\Lib\site-packages\scipy\stats\_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [35]:
paper_agreement = dfagreement.copy()

In [36]:
num_cols = ["Mean (Multi)", "Mean (Single)", "Δ (M−S)", "t", "p", "q_FDR"]

In [37]:
for c in num_cols:
    paper_agreement[c] = pd.to_numeric(paper_agreement[c], errors="coerce")

In [38]:
def highlight_p(val):
    # robust: handle strings like "0.123"
    try:
        v = float(val)
    except (TypeError, ValueError):
        return ""
    if pd.isna(v):
        return ""
    if v < 0.01:
        return "font-weight:bold; background-color:#c6f6d5;"  
    elif v < 0.05:
        return "font-weight:bold; background-color:#fff2a8;" 
    elif v < 0.10:
        return "font-weight:bold; background-color:#d9f2ff;"  
    return ""

In [39]:
paper_agreement[num_cols] = paper_agreement[num_cols].round(3)
paper_agreement

,Domain,Mean (Multi),Mean (Single),Δ (M−S),t,p,q_FDR
0,EV,0.280,0.320,-0.040,-0.577,0.580,0.696
1,EVN,0.337,0.495,-0.158,-1.970,0.087,0.238
2,HWB,0.440,0.360,0.080,0.756,0.471,0.696
3,PCPR,0.238,0.487,-0.249,-2.843,0.022,0.132
4,PIPP,0.490,0.405,0.085,0.891,0.399,0.684
5,POC,0.800,0.880,-0.080,-0.459,0.663,0.723
6,POM,0.250,0.650,-0.400,-2.499,0.047,0.188
7,POS,0.771,0.371,0.400,3.300,0.012,0.132
8,POST,0.360,0.480,-0.120,-0.671,0.522,0.696
9,RV,1.000,0.800,0.200,2.138,0.099,0.238


In [40]:
df_VE = (df_all
              .groupby(["country", "domain_grp", "group"], as_index=False)["variance_explained"]
              .mean())

In [41]:
ve_wide = (df_VE
    .pivot_table(index=["country", "domain_grp"],
                 columns="group",
                 values="variance_explained",
                 aggfunc="mean")     
    .reset_index()
)

In [42]:
ve_wide["ΔVE"] = ve_wide["llm"] - ve_wide["human"]

In [43]:
ve_wide = ve_wide[["country", "domain_grp", "ΔVE"]]

In [44]:
rows_VE = []
for dom, sub in ve_wide.groupby("domain_grp"):
    def culture_of(c):
        if c in single_culture:
            return "Single"
        if c in multi_culture:
            return "Multi"
        return np.nan

    sub["culture"] = sub["country"].map(culture_of)
    
    x = sub.loc[sub["culture"].eq("Multi"), "ΔVE"].dropna().to_numpy()
    y = sub.loc[sub["culture"].eq("Single"), "ΔVE"].dropna().to_numpy()

    t, p = ttest_ind(x, y, equal_var=False, nan_policy="omit")
    rows_VE.append({
        "Domain": dom,
        "Mean (Multi)": np.mean(x) if len(x) else np.nan,
        "Mean (Single)": np.mean(y) if len(y) else np.nan,
        "Δ (M−S)": (np.mean(x) - np.mean(y)) if (len(x) and len(y)) else np.nan,
        "t": t,
        "p": p
    })

dfΔVE = pd.DataFrame(rows_VE).sort_values("Domain").reset_index(drop=True)
dfΔVE["q_FDR"] = multipletests(dfΔVE["p"], method="fdr_bh")[1]

In [45]:
paper_ΔVE = dfΔVE.copy()

In [46]:
for c in num_cols:
    paper_ΔVE[c] = pd.to_numeric(paper_ΔVE[c], errors="coerce")

In [47]:
paper_ΔVE[num_cols] = paper_ΔVE[num_cols].round(3)
paper_ΔVE

,Domain,Mean (Multi),Mean (Single),Δ (M−S),t,p,q_FDR
0,EV,0.139,0.082,0.057,0.866,0.424,0.509
1,EVN,-0.101,-0.216,0.115,1.408,0.200,0.400
2,HWB,-0.031,-0.305,0.274,4.254,0.003,0.036
3,PCPR,0.150,0.040,0.111,3.418,0.017,0.093
4,PIPP,0.071,0.034,0.037,1.774,0.115,0.275
5,POC,0.228,0.121,0.107,1.059,0.322,0.484
6,POM,0.121,0.085,0.037,0.772,0.471,0.514
7,POS,0.080,-0.071,0.150,3.450,0.023,0.093
8,POST,0.202,0.150,0.051,0.957,0.389,0.509
9,RV,0.167,0.176,-0.009,-0.109,0.917,0.917
